# AG-YOLOv11Net: Integrated Smart Parking Implementation

**Notebook implementation based on the uploaded 91-page paper** *An Integrated Smart Parking Framework for Accurate Parking Slot Detection Occupancy Classification and Cyber Intrusion Detection*.

This notebook implements the three connected modules:
1. **Parking-slot detection:** Ultralytics YOLO11 with a MobileNetV2-derived lightweight feature branch/attention-ready integration, CLAHE preprocessing and detection evaluation.
2. **Occupancy classification:** MobileNetV2 feature extraction + an ISOA-style optimizer (chaotic initialization, adaptive exploration/exploitation and regeneration) for classifier weights and threshold.
3. **Cyber intrusion detection:** Res-GhostNet (Ghost feature generation, residual depthwise/pointwise blocks and channel attention) trained on CIC-IDS2017 flow CSVs.

The paper specifies the Parking Space Detection Dataset, CNRPark+EXT and CIC-IDS2017; 70/15/15 train/validation/test splits; and metrics including mAP@0.5, occupancy Accuracy/Precision/Recall/F1/MCC/Kappa and IDS Accuracy/Precision/Recall/F1/latency/throughput/resource use.

**Important reproducibility note:** The paper does not provide the datasets inside the PDF. Set the dataset paths below to your own downloaded copies. The notebook never fabricates samples or labels. The YOLO detector uses the supported YOLO11 training implementation; it is not a byte-for-byte recreation of a proprietary modified YOLO11 backbone. The MobileNetV2-attention feature extractor and the ISOA occupancy optimizer and Res-GhostNet IDS are implemented explicitly here. Reported paper scores are reference values, not results produced by this notebook.

## 0. Runtime and dataset preparation

Recommended: Google Colab with GPU runtime. Mount Drive and point the configuration variables to the extracted datasets.

**Detection dataset format:** standard YOLO layout (`images/train`, `images/val`, `images/test`, corresponding `labels/...`, and `data.yaml`). Labels must contain class id and normalized `x_center y_center width height`. Include a class representing a parking slot.

**Occupancy dataset format (choose one):**
- ROI folders: `occupancy_root/train/occupied`, `occupancy_root/train/vacant` (and optional `val`, `test`); or
- CSV with `image_path` and `label` columns, where label is `occupied` or `vacant`.

**IDS dataset:** CIC-IDS2017 extracted CSV files, with a `Label` column. CSVs may be in one directory. Use CICFlowMeter flow features, not raw packet payloads.

In [ ]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies (run once per fresh Colab runtime)
!pip -q install ultralytics tensorflow opencv-python-headless scikit-learn pandas numpy matplotlib seaborn pyyaml joblib

In [ ]:
import os, glob, json, math, time, random, warnings, gc
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score)
from sklearn.utils.class_weight import compute_class_weight
from ultralytics import YOLO
warnings.filterwarnings('ignore')
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

In [ ]:
# EDIT THESE PATHS
ROOT = Path('/content/drive/MyDrive/AG_YOLOv11Net')
DETECTION_ROOT = ROOT / 'ParkingSpaceDetection_YOLO'  # YOLO-format dataset
OCCUPANCY_ROOT = ROOT / 'CNRPark_OCCUPANCY_ROI'       # class folders or CSV
OCCUPANCY_CSV = ROOT / 'occupancy_labels.csv'         # optional; set None if not used
IDS_ROOT = ROOT / 'CICIDS2017'                        # extracted CIC-IDS2017 CSV directory
OUTPUT_DIR = Path('/content/AG_YOLOv11Net_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE=640
BATCH_SIZE=16
DETECTION_EPOCHS=50
OCC_EPOCHS=15
IDS_EPOCHS=20
ISOA_POPULATION=12
ISOA_ITERATIONS=25
print('Detection root exists:', DETECTION_ROOT.exists())
print('Occupancy root exists:', OCCUPANCY_ROOT.exists())
print('IDS root exists:', IDS_ROOT.exists())

## 1. Dataset validation and CLAHE preprocessing

The paper applies CLAHE, geometric augmentation, noise perturbation, resizing to 640×640 and normalization. YOLO's online augmentation is used during training; CLAHE is available as a deterministic image enhancement operation and can be applied to a copy of the image dataset if desired.

In [ ]:
def clahe_bgr(image, clip_limit=2.0, tile_grid=(8,8)):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid).apply(l)
    return cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2BGR)

def preview_clahe(image_path):
    im=cv2.imread(str(image_path))
    if im is None: raise FileNotFoundError(image_path)
    out=clahe_bgr(im)
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    ax[0].imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB)); ax[0].set_title('Original')
    ax[1].imshow(cv2.cvtColor(out,cv2.COLOR_BGR2RGB)); ax[1].set_title('CLAHE')
    for a in ax: a.axis('off')
    plt.show()

def validate_yolo_dataset(root):
    root=Path(root)
    yaml_path=root/'data.yaml'
    required=[root/'images/train',root/'images/val',root/'labels/train',root/'labels/val']
    missing=[str(p) for p in required if not p.exists()]
    if not yaml_path.exists(): missing.append(str(yaml_path))
    if missing:
        raise FileNotFoundError('YOLO dataset structure incomplete. Missing: '+', '.join(missing))
    for split in ['train','val']:
        imgs=list((root/'images'/split).glob('*'))
        imgs=[p for p in imgs if p.suffix.lower() in ['.jpg','.jpeg','.png','.bmp']]
        if not imgs: raise ValueError(f'No images found in images/{split}')
        print(split, 'images:', len(imgs))
    return yaml_path

if DETECTION_ROOT.exists():
    yolo_yaml=validate_yolo_dataset(DETECTION_ROOT)
    sample=next(iter((DETECTION_ROOT/'images/train').glob('*')))
    preview_clahe(sample)
else:
    print('Set DETECTION_ROOT to the extracted YOLO-format Parking Space Detection Dataset.')

## 2. Attention-guided YOLO11 parking-slot detection

The paper's detector uses YOLOv11 with a MobileNetV2 lightweight backbone, multi-scale C3k2/SPPF/C2PSA refinement and an attention-guided detection head. This executable notebook trains the official YOLO11 detector (`yolo11n.pt` by default) with edge-friendly settings and exposes MobileNetV2-attention features for the separate occupancy classifier. For an exact custom YOLO11 graph, a separately authored Ultralytics model YAML/weight conversion is required; it is not silently claimed here.

In [ ]:
if DETECTION_ROOT.exists():
    # Verify YAML class names and paths before training
    import yaml
    with open(DETECTION_ROOT/'data.yaml','r') as f: det_cfg=yaml.safe_load(f)
    print(det_cfg)
    detector = YOLO('yolo11n.pt')  # lightweight YOLO11 baseline
    det_train_start=time.time()
    det_results=detector.train(
        data=str(DETECTION_ROOT/'data.yaml'),
        imgsz=IMG_SIZE, epochs=DETECTION_EPOCHS, batch=BATCH_SIZE,
        device=0 if tf.config.list_physical_devices('GPU') else 'cpu',
        workers=2, patience=10, project=str(OUTPUT_DIR/'detection'),
        name='ag_yolov11net', exist_ok=True, seed=SEED,
        pretrained=True, optimizer='AdamW', lr0=1e-3,
        cos_lr=True, close_mosaic=10,
        degrees=8, translate=0.10, scale=0.35, fliplr=0.5,
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.35,
        mixup=0.05, mosaic=0.7, verbose=False
    )
    detection_train_seconds=time.time()-det_train_start
    best_detector_path=Path(detector.trainer.best)
    print('Best detector:', best_detector_path)
    print('Training seconds:', round(detection_train_seconds,2))
else:
    detector=None
    print('Detection training skipped: configure DETECTION_ROOT.')

In [ ]:
# Detection validation metrics (Ultralytics reports precision, recall, mAP50, mAP50-95)
if detector is not None:
    det_val=detector.val(data=str(DETECTION_ROOT/'data.yaml'), imgsz=IMG_SIZE, verbose=False)
    print('Detection metrics object:', det_val.results_dict)

## 3. MobileNetV2-attention ROI feature extraction

Detected parking-slot crops are classified as occupied/vacant. For supervised training, supply labeled ROI images. MobileNetV2 provides compact embeddings; channel attention recalibrates the feature vector. The ISOA optimizer later tunes a logistic classifier's feature weights and decision threshold offline.

In [ ]:
# MobileNetV2 feature extractor with lightweight channel attention
base=tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3), include_top=False, weights='imagenet', pooling='avg'
)
base.trainable=False
inp=tf.keras.Input((224,224,3), name='roi_image')
x=tf.keras.applications.mobilenet_v2.preprocess_input(inp)
f=base(x, training=False)
att=tf.keras.layers.Dense(max(8, f.shape[-1]//8), activation='relu')(f)
att=tf.keras.layers.Dense(f.shape[-1], activation='sigmoid')(att)
f=tf.keras.layers.Multiply()([f,att])
embedding_model=tf.keras.Model(inp,f,name='MobileNetV2_Attention_Embedding')
print('Embedding dimension:', embedding_model.output_shape[-1])

In [ ]:
# Load occupancy ROI dataset without inventing labels
IMG_EXTS={'.jpg','.jpeg','.png','.bmp'}
def load_occupancy_manifest(root, csv_path=None):
    rows=[]
    if csv_path and Path(csv_path).exists():
        df=pd.read_csv(csv_path)
        if not {'image_path','label'}.issubset(df.columns):
            raise ValueError('Occupancy CSV must contain image_path and label columns.')
        for _,r in df.iterrows():
            p=Path(str(r.image_path))
            if not p.is_absolute(): p=Path(root)/p
            if p.exists(): rows.append((str(p),str(r.label).strip().lower()))
    else:
        root=Path(root)
        for split in ['train','val','test']:
            splitdir=root/split
            if not splitdir.exists(): continue
            for clsdir in splitdir.iterdir():
                if clsdir.is_dir():
                    label=clsdir.name.lower()
                    for p in clsdir.rglob('*'):
                        if p.suffix.lower() in IMG_EXTS: rows.append((str(p),label))
        if not rows:
            for clsdir in root.iterdir() if root.exists() else []:
                if clsdir.is_dir() and clsdir.name.lower() in ['occupied','vacant','empty','free']:
                    label='vacant' if clsdir.name.lower() in ['vacant','empty','free'] else 'occupied'
                    for p in clsdir.rglob('*'):
                        if p.suffix.lower() in IMG_EXTS: rows.append((str(p),label))
    df=pd.DataFrame(rows,columns=['image_path','label']).drop_duplicates()
    if len(df):
        df['label']=df.label.replace({'empty':'vacant','free':'vacant','available':'vacant','full':'occupied'})
        df=df[df.label.isin(['occupied','vacant'])]
    return df.reset_index(drop=True)

occ_df=load_occupancy_manifest(OCCUPANCY_ROOT, str(OCCUPANCY_CSV) if OCCUPANCY_CSV else None)
print('Occupancy ROI samples:',len(occ_df))
if len(occ_df): print(occ_df.label.value_counts())
else: print('No labeled ROI images found. Configure OCCUPANCY_CSV or class-folder structure.')

In [ ]:
def image_to_embedding(path):
    im=cv2.imread(str(path))
    if im is None: return None
    im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
    im=cv2.resize(im,(224,224),interpolation=cv2.INTER_AREA)
    return embedding_model.predict(np.expand_dims(im.astype('float32'),0),verbose=0)[0]

if len(occ_df):
    X_occ=[]; y_occ=[]
    for p,l in zip(occ_df.image_path,occ_df.label):
        e=image_to_embedding(p)
        if e is not None: X_occ.append(e); y_occ.append(1 if l=='occupied' else 0)
    X_occ=np.asarray(X_occ,dtype='float32'); y_occ=np.asarray(y_occ,dtype='int32')
    X_train_occ,X_tmp_occ,y_train_occ,y_tmp_occ=train_test_split(
        X_occ,y_occ,test_size=0.30,random_state=SEED,stratify=y_occ)
    X_val_occ,X_test_occ,y_val_occ,y_test_occ=train_test_split(
        X_tmp_occ,y_tmp_occ,test_size=0.50,random_state=SEED,stratify=y_tmp_occ)
    print(X_train_occ.shape,X_val_occ.shape,X_test_occ.shape)
else:
    X_occ=np.empty((0,1280)); y_occ=np.array([],dtype=int)

## 4. Improved Starfish Optimization Algorithm (ISOA)

A binary logistic occupancy classifier is parameterized by feature weights, bias and a decision threshold. Fitness combines validation accuracy, precision, recall, F1 and loss. The optimizer uses chaotic circle-map initialization, multi-dimension exploration, adaptive exploration energy and regeneration-style exploitation. Optimization is offline; inference uses only the selected parameters.

In [ ]:
def sigmoid_np(z): return 1.0/(1.0+np.exp(-np.clip(z,-35,35)))

def isoaclassifier_predict(X, sol):
    d=X.shape[1]
    w=sol[:d]; b=sol[d]; threshold=np.clip(sol[d+1],0.05,0.95)
    return (sigmoid_np(X@w+b)>=threshold).astype(int)

def occ_fitness(sol, X, y):
    pred=isoaclassifier_predict(X,sol)
    acc=accuracy_score(y,pred)
    prec=precision_score(y,pred,zero_division=0)
    rec=recall_score(y,pred,zero_division=0)
    f1=f1_score(y,pred,zero_division=0)
    prob=sigmoid_np(X@sol[:X.shape[1]]+sol[X.shape[1]])
    eps=1e-7
    loss=-np.mean(y*np.log(prob+eps)+(1-y)*np.log(1-prob+eps))
    return 0.25*acc+0.20*prec+0.20*rec+0.25*f1-0.10*loss

def chaotic_circle_init(pop, dim, low, high, rng):
    z=rng.random((pop,dim))
    for _ in range(5):
        z=(z+0.2-0.5/(2*np.pi)*np.sin(2*np.pi*z))%1.0
    return low+z*(high-low)

def run_ISOA(Xtr,ytr,Xv,yv,pop_size=12,max_iter=25):
    rng=np.random.default_rng(SEED)
    d=Xtr.shape[1]+2
    low=np.full(d,-2.0); high=np.full(d,2.0)
    low[-1]=0.15; high[-1]=0.85
    pop=chaotic_circle_init(pop_size,d,low,high,rng)
    fit=np.array([occ_fitness(s,Xv,yv) for s in pop])
    best=pop[np.argmax(fit)].copy(); bestfit=float(np.max(fit))
    history=[]
    for t in range(max_iter):
        energy=1.0-t/max_iter
        newpop=pop.copy()
        for i in range(pop_size):
            candidate=pop[i].copy()
            if d>5:
                dims=rng.choice(d,size=5,replace=False)
            else: dims=np.array([rng.integers(d)])
            if rng.random()<0.65:
                angle=2*np.pi*rng.random()
                candidate[dims]=candidate[dims]+energy*np.cos(angle)*(best[dims]-candidate[dims])+0.15*energy*np.sin(angle)*rng.normal(size=len(dims))
            else:
                j,k=rng.choice(pop_size,2,replace=False)
                candidate[dims]=candidate[dims]+rng.random()*(pop[j,dims]-pop[k,dims])+0.25*energy*(best[dims]-candidate[dims])
            # Regeneration-inspired local exploitation around current global best
            if rng.random()<0.20:
                candidate=best+ rng.normal(0,0.08*(1-energy+0.05),size=d)
            candidate=np.clip(candidate,low,high)
            newpop[i]=candidate
        newfit=np.array([occ_fitness(s,Xv,yv) for s in newpop])
        improved=newfit>fit
        pop[improved]=newpop[improved]; fit[improved]=newfit[improved]
        idx=int(np.argmax(fit))
        if fit[idx]>bestfit: bestfit=float(fit[idx]); best=pop[idx].copy()
        history.append(bestfit)
    return best,history

if len(X_occ):
    iso_best,isoa_history=run_ISOA(X_train_occ,y_train_occ,X_val_occ,y_val_occ,ISOA_POPULATION,ISOA_ITERATIONS)
    plt.figure(figsize=(7,4)); plt.plot(isoa_history); plt.xlabel('Iteration'); plt.ylabel('Best validation fitness'); plt.title('ISOA convergence'); plt.grid(True); plt.show()
else:
    iso_best=None

In [ ]:
# Occupancy test evaluation
if iso_best is not None:
    y_occ_pred=isoaclassifier_predict(X_test_occ,iso_best)
    occ_metrics={
        'Accuracy':accuracy_score(y_test_occ,y_occ_pred),
        'Precision':precision_score(y_test_occ,y_occ_pred,zero_division=0),
        'Recall':recall_score(y_test_occ,y_occ_pred,zero_division=0),
        'F1':f1_score(y_test_occ,y_occ_pred,zero_division=0),
        'MCC':matthews_corrcoef(y_test_occ,y_occ_pred),
        'Kappa':cohen_kappa_score(y_test_occ,y_occ_pred)
    }
    print(pd.Series(occ_metrics).to_string())
    print(classification_report(y_test_occ,y_occ_pred,target_names=['Vacant','Occupied'],zero_division=0))
    print('Confusion matrix:\n',confusion_matrix(y_test_occ,y_occ_pred))

## 5. CIC-IDS2017 preprocessing

The paper identifies CIC-IDS2017 as the IDS benchmark, with CICFlowMeter statistical flow features and multiple attack families. This cell reads all CSV files recursively, removes invalid/infinite values, normalizes column names, and uses the `Label` column. Duplicate and identifier-like columns are removed to reduce leakage. Labels remain multiclass; the model predicts Normal and attack categories.

In [ ]:
def load_cicids_csvs(root):
    files=list(Path(root).rglob('*.csv'))
    if not files: raise FileNotFoundError(f'No CIC-IDS2017 CSV files found under {root}')
    frames=[]
    for f in files:
        try:
            part=pd.read_csv(f,low_memory=False,encoding_errors='ignore')
            part.columns=[str(c).strip() for c in part.columns]
            if 'Label' in part.columns or ' Label' in part.columns:
                part['__source_file']=f.name
                frames.append(part)
                print(f.name, part.shape)
        except Exception as e: print('Skipping',f.name,repr(e))
    if not frames: raise ValueError('No CSV with a Label column found. Ensure CICFlowMeter CSV files are extracted.')
    return pd.concat(frames,ignore_index=True)

if IDS_ROOT.exists():
    ids_df=load_cicids_csvs(IDS_ROOT)
    ids_df.columns=[c.strip() for c in ids_df.columns]
    print(ids_df['Label'].value_counts().head(20))
else:
    ids_df=None
    print('Set IDS_ROOT to CIC-IDS2017 CSV directory.')

In [ ]:
def prepare_ids_data(df, max_rows=None):
    df=df.copy()
    df.columns=[str(c).strip() for c in df.columns]
    label_col='Label' if 'Label' in df.columns else next(c for c in df.columns if c.lower()=='label')
    y=df[label_col].astype(str).str.strip()
    X=df.drop(columns=[label_col],errors='ignore')
    X=X.drop(columns=[c for c in X.columns if c.startswith('__')],errors='ignore')
    # Remove identifiers, timestamps and constant columns; keep numeric CICFlowMeter features.
    X=X.drop(columns=[c for c in X.columns if any(k in c.lower() for k in ['flow id','source ip','destination ip','timestamp'])],errors='ignore')
    for c in X.columns: X[c]=pd.to_numeric(X[c],errors='coerce')
    X=X.replace([np.inf,-np.inf],np.nan)
    X=X.dropna(axis=1,how='all')
    X=X.fillna(X.median(numeric_only=True)).fillna(0)
    nunique=X.nunique(dropna=False)
    X=X.loc[:,nunique>1]
    # Avoid accidentally overwhelming memory; deterministic stratified cap if configured.
    if max_rows and len(X)>max_rows:
        take,_=train_test_split(np.arange(len(X)),train_size=max_rows,random_state=SEED,stratify=y)
        X=X.iloc[take]; y=y.iloc[take]
    le=LabelEncoder()
    y_enc=le.fit_transform(y)
    X_train,X_temp,y_train,y_temp=train_test_split(X.values,y_enc,test_size=0.30,random_state=SEED,stratify=y_enc)
    X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=0.50,random_state=SEED,stratify=y_temp)
    scaler=StandardScaler()
    X_train=scaler.fit_transform(X_train).astype('float32')
    X_val=scaler.transform(X_val).astype('float32')
    X_test=scaler.transform(X_test).astype('float32')
    return X_train,X_val,X_test,y_train,y_val,y_test,scaler,le,list(X.columns)

if ids_df is not None:
    X_train_ids,X_val_ids,X_test_ids,y_train_ids,y_val_ids,y_test_ids,ids_scaler,ids_encoder,ids_features=prepare_ids_data(ids_df, max_rows=None)
    print('IDS shapes:',X_train_ids.shape,X_val_ids.shape,X_test_ids.shape)
    print('Classes:',list(ids_encoder.classes_))

## 6. Res-GhostNet intrusion detection model

Res-GhostNet follows the paper's Ghost feature generation → residual feature learning → attention refinement → fully connected Softmax classifier. CICFlowMeter vectors are treated as a one-dimensional feature sequence. Training uses categorical cross-entropy and Adam; inference is a single forward pass.

In [ ]:
def ghost_1d(x, filters, ratio=2, kernel_size=3, name='ghost'):
    primary=max(1,filters//ratio)
    x0=tf.keras.layers.Conv1D(primary,1,padding='same',use_bias=False,name=name+'_primary')(x)
    x0=tf.keras.layers.BatchNormalization()(x0)
    x0=tf.keras.layers.Activation('swish')(x0)
    cheap=tf.keras.layers.DepthwiseConv1D(kernel_size,padding='same',depth_multiplier=max(1,filters//primary-1),use_bias=False,name=name+'_cheap')(x0)
    cheap=tf.keras.layers.BatchNormalization()(cheap)
    out=tf.keras.layers.Concatenate()([x0,cheap])
    return tf.keras.layers.Conv1D(filters,1,padding='same',name=name+'_trim')(out)

def residual_ghost_block(x, filters, name):
    shortcut=x
    y=ghost_1d(x,filters,name=name+'_ghost')
    y=tf.keras.layers.Conv1D(filters,1,padding='same',use_bias=False,name=name+'_expand')(y)
    y=tf.keras.layers.BatchNormalization()(y); y=tf.keras.layers.Activation('swish')(y)
    y=tf.keras.layers.DepthwiseConv1D(3,padding='same',use_bias=False,name=name+'_dw')(y)
    y=tf.keras.layers.BatchNormalization()(y); y=tf.keras.layers.Activation('swish')(y)
    y=tf.keras.layers.Conv1D(filters,1,padding='same',use_bias=False,name=name+'_project')(y)
    y=tf.keras.layers.BatchNormalization()(y)
    if int(shortcut.shape[-1])!=filters:
        shortcut=tf.keras.layers.Conv1D(filters,1,padding='same',use_bias=False)(shortcut)
        shortcut=tf.keras.layers.BatchNormalization()(shortcut)
    y=tf.keras.layers.Add()([shortcut,y])
    return tf.keras.layers.Activation('swish')(y)

def build_res_ghostnet(n_features,n_classes):
    inp=tf.keras.Input((n_features,),name='flow_features')
    x=tf.keras.layers.Reshape((n_features,1))(inp)
    x=residual_ghost_block(x,32,'rg1')
    x=tf.keras.layers.MaxPooling1D(2,padding='same')(x)
    x=residual_ghost_block(x,64,'rg2')
    # Channel attention
    a=tf.keras.layers.GlobalAveragePooling1D()(x)
    a=tf.keras.layers.Dense(16,activation='relu')(a)
    a=tf.keras.layers.Dense(64,activation='sigmoid')(a)
    a=tf.keras.layers.Reshape((1,64))(a)
    x=tf.keras.layers.Multiply()([x,a])
    x=tf.keras.layers.GlobalAveragePooling1D()(x)
    x=tf.keras.layers.Dense(64,activation='swish')(x)
    x=tf.keras.layers.Dropout(0.30)(x)
    out=tf.keras.layers.Dense(n_classes,activation='softmax',name='intrusion_class')(x)
    model=tf.keras.Model(inp,out,name='Res_GhostNet_IDS')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model

if ids_df is not None:
    ids_model=build_res_ghostnet(X_train_ids.shape[1],len(ids_encoder.classes_))
    ids_model.summary()
    classes=np.unique(y_train_ids)
    weights=compute_class_weight(class_weight='balanced',classes=classes,y=y_train_ids)
    class_weights={int(c):float(w) for c,w in zip(classes,weights)}
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=4,restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=2,min_lr=1e-6)
    ]
    ids_history=ids_model.fit(X_train_ids,y_train_ids,validation_data=(X_val_ids,y_val_ids),
        epochs=IDS_EPOCHS,batch_size=512,class_weight=class_weights,callbacks=callbacks,verbose=1)
else:
    ids_model=None

In [ ]:
# IDS held-out test metrics and operational measurements
if ids_model is not None:
    t0=time.perf_counter()
    ids_prob=ids_model.predict(X_test_ids,batch_size=1024,verbose=0)
    elapsed=time.perf_counter()-t0
    ids_pred=np.argmax(ids_prob,axis=1)
    ids_metrics={
        'Accuracy':accuracy_score(y_test_ids,ids_pred),
        'Precision_macro':precision_score(y_test_ids,ids_pred,average='macro',zero_division=0),
        'Recall_macro':recall_score(y_test_ids,ids_pred,average='macro',zero_division=0),
        'F1_macro':f1_score(y_test_ids,ids_pred,average='macro',zero_division=0),
        'MCC':matthews_corrcoef(y_test_ids,ids_pred),
        'Latency_ms_per_flow':1000*elapsed/max(1,len(X_test_ids)),
        'Throughput_flows_per_second':len(X_test_ids)/max(elapsed,1e-9)
    }
    print(pd.Series(ids_metrics).to_string())
    print(classification_report(y_test_ids,ids_pred,labels=np.arange(len(ids_encoder.classes_)),
        target_names=ids_encoder.classes_,zero_division=0))
    print('Confusion matrix shape:',confusion_matrix(y_test_ids,ids_pred).shape)

## 7. End-to-end parking inference and security output

This cell runs YOLO detection on a parking image and independently predicts network-flow security state. The detector returns slot/object boxes according to the trained detection dataset's class definitions. Occupancy classification requires ROI crops with the same semantics as the labeled occupancy training data. The IDS receives flow feature vectors, not image pixels.

In [ ]:
def predict_parking_image(image_path, conf=0.25):
    if detector is None: raise RuntimeError('Train/load detector first.')
    result=detector.predict(source=str(image_path),imgsz=IMG_SIZE,conf=conf,verbose=False)[0]
    plotted=result.plot()
    plt.figure(figsize=(12,8)); plt.imshow(cv2.cvtColor(plotted,cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
    records=[]
    if result.boxes is not None:
        for b in result.boxes:
            xyxy=b.xyxy[0].cpu().numpy().tolist()
            cls=int(b.cls[0].cpu().numpy())
            records.append({'box_xyxy':xyxy,'class_id':cls,'confidence':float(b.conf[0].cpu().numpy())})
    return records

def predict_intrusion_flow(flow_dataframe):
    if ids_model is None: raise RuntimeError('Train IDS model first.')
    row=flow_dataframe.copy()
    row=row.reindex(columns=ids_features,fill_value=0)
    for c in row.columns: row[c]=pd.to_numeric(row[c],errors='coerce')
    row=row.replace([np.inf,-np.inf],np.nan).fillna(0)
    xx=ids_scaler.transform(row.values).astype('float32')
    p=ids_model.predict(xx,verbose=0)[0]
    idx=int(np.argmax(p))
    return {'predicted_class':str(ids_encoder.classes_[idx]),'confidence':float(p[idx])}

# Example use after training:
# parking_result = predict_parking_image('/path/to/parking_image.jpg')
# intrusion_result = predict_intrusion_flow(one_flow_dataframe)

## 8. Save trained artifacts and experiment report

Only trained model artifacts and fitted preprocessing objects are saved. No fabricated metric values are written. If a module was not trained because its dataset was not configured, its result remains absent.

In [ ]:
import joblib
ARTIFACT_DIR=OUTPUT_DIR/'artifacts'
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
if detector is not None:
    print('Detector best weights:',detector.trainer.best)
if iso_best is not None:
    np.save(ARTIFACT_DIR/'isoa_occupancy_solution.npy',iso_best)
    joblib.dump(embedding_model,ARTIFACT_DIR/'mobilenetv2_attention_embedding.keras')
if ids_model is not None:
    ids_model.save(ARTIFACT_DIR/'res_ghostnet_ids.keras')
    joblib.dump({'scaler':ids_scaler,'label_encoder':ids_encoder,'feature_names':ids_features},
                ARTIFACT_DIR/'ids_preprocessing.joblib')
report={
    'framework':'AG-YOLOv11Net implementation',
    'detection_trained':detector is not None,
    'occupancy_trained':iso_best is not None,
    'ids_trained':ids_model is not None,
    'detection_metrics':det_val.results_dict if detector is not None else None,
    'occupancy_metrics':occ_metrics if iso_best is not None else None,
    'ids_metrics':ids_metrics if ids_model is not None else None,
    'seed':SEED
}
with open(OUTPUT_DIR/'experiment_report.json','w') as f: json.dump(report,f,indent=2,default=float)
print(json.dumps(report,indent=2,default=float))

## 9. Interpretation and reproducibility checklist

- Keep the test split untouched until final evaluation. The notebook uses 70/15/15 partitions for IDS and requires pre-separated YOLO train/validation/test directories (Ultralytics validation uses the YAML validation split).
- The paper's parking datasets are **Parking Space Detection Dataset** and **CNRPark+EXT**; the IDS dataset is **CIC-IDS2017**. Detection annotations must be converted to YOLO format before training. Occupancy labels must identify each ROI as occupied/vacant.
- For cross-dataset generalization, train the detector/occupancy model on one parking dataset and evaluate on the other without fine-tuning; ensure class definitions and annotation conventions match.
- The paper reports 99.28% overall detection accuracy, 98.26% F1, 97.53% mAP and 99.33% IDS accuracy in its abstract. These are paper-reported reference figures and must not be presented as reproduced results unless your own held-out experiments achieve them.
- The paper's architecture diagram on page 12 shows the vision pipeline (CLAHE → MobileNetV2-attention/Yolo neck → C3k2/SPPF/C2PSA → occupancy/IDS); the IDS diagram on page 36 depicts Ghost feature generation, residual learning, attention refinement and classification.